In [ ]:
import os
import glob
import time
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

class MacenkoNormalizer:
    def __init__(self):
        # 默认参考值 (会在 fit 中被覆盖)
        self.HERef = np.array([[0.5626, 0.2159, 0.4062],
                               [0.7201, 0.8012, 0.5581]])
        self.maxCRef = np.array([1.9705, 1.0308])

    def fit(self, target):
        """ 获取参考图片的染色特征矩阵 """
        HE, maxC = self.get_stain_matrix(target)
        if HE is not None:
            self.HERef = HE
            self.maxCRef = maxC
            print("✅ 参考图片特征提取完毕")
        else:
            print("❌ 参考图片特征提取失败（可能是背景太多或颜色太浅），请更换参考图！")

    def normalize(self, src):
        """ 对输入图片进行归一化 """
        h, w, c = src.shape
        stain_matrix_src, maxC_src = self.get_stain_matrix(src)
        
        # 异常处理：提取失败直接返回原图
        if stain_matrix_src is None:
            return src

        # 转换到 OD 空间
        OD = -np.log((src.astype(float) + 1) / 255)
        OD = OD.reshape((-1, 3))

        # 计算源图像的浓度
        # OD(N,3) dot pinv(HE)(3,2) -> C(N,2)
        try:
            source_concentration = np.dot(OD, np.linalg.pinv(stain_matrix_src))
        except:
            return src

        # 归一化浓度
        source_concentration *= (self.maxCRef / maxC_src)

        # 重建图像
        # C(N,2) dot HE(2,3) -> OD(N,3)
        resp = np.dot(source_concentration, self.HERef)
        
        # OD 转回 RGB
        resp = resp.reshape((h, w, 3))
        resp = np.exp(-resp) * 255
        resp = np.clip(resp, 0, 255).astype(np.uint8)
        
        return resp

    def get_stain_matrix(self, I, beta=0.15, alpha=1):
        """ 核心算法：提取染色矩阵 """
        I = I.reshape((-1, 3))
        OD = -np.log((I.astype(float) + 1) / 255)
        
        # 移除透明度过低的像素 (去背景)
        ODhat = OD[(OD > beta).all(axis=1)]
        
        # 如果有效像素太少 (比如全是白色背景)，返回 None
        if ODhat.shape[0] < 100: 
            return None, None

        # 计算特征向量 (SVD)
        try:
            cov = np.cov(ODhat.T)
            eig_vals, eig_vecs = np.linalg.eigh(cov)
        except:
            return None, None
            
        eig_vecs = eig_vecs[:, [2, 1]] # 取最大的两个特征向量

        # 保证方向一致性
        if eig_vecs[0, 0] < 0: eig_vecs[:, 0] *= -1
        if eig_vecs[0, 1] < 0: eig_vecs[:, 1] *= -1

        # 投影到特征向量平面
        That = np.dot(ODhat, eig_vecs)
        phi = np.arctan2(That[:, 1], That[:, 0])

        minPhi = np.percentile(phi, alpha)
        maxPhi = np.percentile(phi, 100 - alpha)

        vMin = np.dot(eig_vecs, np.array([np.cos(minPhi), np.sin(minPhi)]))
        vMax = np.dot(eig_vecs, np.array([np.cos(maxPhi), np.sin(maxPhi)]))

        # 排序：He (小角度) 先, Eo (大角度) 后
        if vMin[0] > vMax[0]:
            HE = np.array([vMin, vMax])
        else:
            HE = np.array([vMax, vMin])

        # === 关键修复：删除了 HE = HE.T ===
        # HE 的形状现在保持 (2, 3) -> 2行(染色剂), 3列(RGB通道)
        
        # 计算最大浓度
        # Y(N,3) dot pinv(HE(2,3)) -> C(N,2)
        # pinv(2,3) -> (3,2)，所以 (N,3) x (3,2) = (N,2) 维度匹配成功
        Y = np.reshape(OD, (-1, 3))
        try:
            C = np.dot(Y, np.linalg.pinv(HE))
        except:
            return None, None
        
        if C.shape[0] == 0:
            return None, None
            
        maxC = np.percentile(C, 99, axis=0)
        
        return HE, maxC

# ================= 主程序 =================

def run_normalization():
    # 1. 配置路径
    input_dir = Path(r"G:\1.Sprouts\High\High.patches.train")
    output_dir = Path(r"G:\1.Sprouts\High\High.patches.train.normalized")
    
    # 参考图片路径 (请确保该图片存在且不仅是白色背景)
    ref_image_path = Path(r"G:\1.Sprouts\High\High.patches.train\TCGA-F4-6854_ROI2_r11_c23_x26903_y48085.png")

    if not ref_image_path.exists():
        print(f"❌ 错误：找不到参考图片 -> {ref_image_path}")
        return
    
    output_dir.mkdir(parents=True, exist_ok=True)

    # 2. 初始化
    print("⏳ 正在分析参考图片...")
    normalizer = MacenkoNormalizer()
    
    ref_img = cv2.imread(str(ref_image_path))
    if ref_img is None:
        print("❌ 无法读取参考图片，请检查文件是否损坏。")
        return
    ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
    
    normalizer.fit(ref_img)

    # 3. 处理循环
    image_files = list(input_dir.glob("*.png"))
    total = len(image_files)
    print(f"📂 找到 {total} 张图片，开始归一化处理...")
    print("-" * 50)

    start_time = time.time()
    count = 0
    success_count = 0
    
    for img_path in image_files:
        count += 1
        save_path = output_dir / img_path.name

        if save_path.exists(): continue

        try:
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            norm_img = normalizer.normalize(img)

            Image.fromarray(norm_img).save(save_path)
            success_count += 1

        except Exception as e:
            print(f"⚠️ 处理失败 {img_path.name}: {e}")

        if count % 100 == 0:
            elapsed = time.time() - start_time
            print(f"   [{count}/{total}] 处理中... (速度: {count/elapsed:.1f} 张/秒)")

    print("-" * 50)
    print(f"🎉 处理完成！成功: {success_count} 张")

if __name__ == "__main__":
    run_normalization()


⏳ 正在分析参考图片...
✅ 参考图片特征提取完毕
📂 找到 38845 张图片，开始归一化处理...
--------------------------------------------------
   [100/38845] 处理中... (速度: 21.2 张/秒)
   [200/38845] 处理中... (速度: 21.2 张/秒)
   [300/38845] 处理中... (速度: 21.2 张/秒)
   [400/38845] 处理中... (速度: 21.3 张/秒)
   [500/38845] 处理中... (速度: 21.3 张/秒)
   [600/38845] 处理中... (速度: 21.3 张/秒)
   [700/38845] 处理中... (速度: 21.3 张/秒)
   [800/38845] 处理中... (速度: 21.4 张/秒)
   [900/38845] 处理中... (速度: 21.8 张/秒)
   [1000/38845] 处理中... (速度: 22.1 张/秒)
   [1100/38845] 处理中... (速度: 22.3 张/秒)
   [1200/38845] 处理中... (速度: 22.4 张/秒)
